In [ ]:
import re
import numpy as np
import pandas as pd
import openpyxl

In [ ]:
xlsx_path = "/content/ghe2021_deaths_whoregion_new2.xlsx"
print(xlsx_path)

/content/ghe2021_deaths_whoregion_new2.xlsx


In [ ]:
# Clean cell text
def normalize_text(x):
    if x is None:
        return None
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x != "" else None


# Fill merged-header style values
def fill_right(values):
    out = []
    last = None
    for v in values:
        if v is not None:
            last = v
        out.append(last)
    return out

In [ ]:
# Identify which cause hierarchy level a row belongs to
def get_level_type_and_cause(row_vals):

    b = normalize_text(row_vals[1])
    c = normalize_text(row_vals[2])
    d = normalize_text(row_vals[3])
    e = normalize_text(row_vals[4])
    f = normalize_text(row_vals[5])

    # All Causes
    if c is not None and c.lower() == "all causes":
        return "all", c

    # Roman numeral
    if b is not None and re.fullmatch(r"[IVXLC]+\.", b) and c is not None:
        return "roman", c

    # Letter A,B,C
    if c is not None and re.fullmatch(r"[A-Z]\.", c) and d is not None:
        return "letter", d

    # Number
    if d is not None and re.fullmatch(r"\d+\.", d) and e is not None:
        return "number", e

    # Lowercase letter
    if e is not None and re.fullmatch(r"[a-z]\.", e) and f is not None:
        return "lower", f

    return None, None

In [ ]:
# Parse worksheet into long format df
def parse_sheet(ws, sheet_name):
    # metadata
    region = normalize_text(ws["F3"].value)
    year = ws["F4"].value

    max_row = ws.max_row
    max_col = ws.max_column

    # header rows
    # row 6 sex
    # row 7 age group
    sex_raw = [normalize_text(ws.cell(6, c).value) for c in range(7, max_col + 1)]
    age_raw = [normalize_text(ws.cell(7, c).value) for c in range(7, max_col + 1)]

    sex_filled = fill_right(sex_raw)
    age_filled = fill_right(age_raw)

    value_columns = []
    for idx, c in enumerate(range(7, max_col + 1)):
        sex = sex_filled[idx]
        age = age_filled[idx]

        if sex is None and age is None:
            continue

        value_columns.append({
            "excel_col": c,
            "sex": sex,
            "age": age
        })

    records = []

    # data starts from row 10
    for r in range(10, max_row + 1):
        row_vals = [ws.cell(r, c).value for c in range(1, 7)]

        if all(v is None for v in row_vals):
            continue

        code = row_vals[0]
        level_type, cause = get_level_type_and_cause(row_vals)
        cause = normalize_text(cause)

        if level_type is None or cause is None:
            continue

        for meta in value_columns:
            c = meta["excel_col"]
            val = ws.cell(r, c).value

            if val is None:
                continue

            try:
                deaths = float(val)
            except:
                continue

            records.append({
                "sheet": sheet_name,
                "region": region,
                "year": int(year),
                "code": code,
                "level_type": level_type,
                "cause": cause,
                "sex": normalize_text(meta["sex"]),
                "age": normalize_text(meta["age"]),
                "deaths": deaths
            })

    df = pd.DataFrame(records)

    # unify text
    df["sex"] = df["sex"].replace({
        "Both sexes": "Both sexes",
        "Male": "Male",
        "Female": "Female"
    })

    df["age"] = df["age"].fillna("Total (all ages)")

    # denominator = all causes deaths for same region/year/sex/age
    all_causes = df[df["level_type"] == "all"].copy()
    all_causes = all_causes[["region", "year", "sex", "age", "deaths"]].rename(
        columns={"deaths": "all_cause_deaths"}
    )

    df = df.merge(
        all_causes,
        on=["region", "year", "sex", "age"],
        how="left"
    )

    df["probability"] = df["deaths"] / df["all_cause_deaths"]

    return df

In [ ]:
# load workbook
wb = openpyxl.load_workbook(xlsx_path, data_only=True)
sheet_names = wb.sheetnames

In [ ]:
# only 2019 sheets
target_sheets = [s for s in sheet_names if re.search(r"\b2019\b", s)]
target_sheets

['Global 2019',
 'AFR 2019',
 'AMR 2019',
 'EMR 2019',
 'EUR 2019',
 'SEAR 2019',
 'WPR 2019']

In [ ]:
# Parse each sheet and combine them
dfs = []
for s in target_sheets:
    ws = wb[s]
    dfs.append(parse_sheet(ws, s))

df_all = pd.concat(dfs, ignore_index=True)
print(df_all.shape)
df_all.head()

(28595, 11)


,sheet,region,year,code,level_type,cause,sex,age,deaths,all_cause_deaths,probability
0,Global 2019,Global,2019,0,all,All Causes,Both sexes,Total (all ages),5.697082e+07,5.697082e+07,1.0
1,Global 2019,Global,2019,0,all,All Causes,Male,Total (all ages),3.073065e+07,3.073065e+07,1.0
2,Global 2019,Global,2019,0,all,All Causes,Female,Total (all ages),2.624017e+07,2.624017e+07,1.0
3,Global 2019,Global,2019,0,all,All Causes,Male,0-28 days,1.398409e+06,1.398409e+06,1.0
4,Global 2019,Global,2019,0,all,All Causes,Male,1-59 months,1.523369e+06,1.523369e+06,1.0


In [ ]:
# keep only the letter layer
df_clean = df_all[
    (df_all["level_type"] == "letter") &
    (df_all["age"] != "Total (all ages)") &
    (df_all["cause"] != "All Causes")
].copy()

print(df_clean.shape)
df_clean.head(20)

(2688, 11)


,sheet,region,year,code,level_type,cause,sex,age,deaths,all_cause_deaths,probability
41,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,0-28 days,61128.945876,1.398409e+06,0.043713
42,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,1-59 months,755223.289826,1.523369e+06,0.495759
43,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,5-14,164937.494206,4.810304e+05,0.342884
44,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,15-29,182901.599178,1.297107e+06,0.141007
45,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,30-49,523998.210087,3.383772e+06,0.154856
46,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,50-59,299531.296122,3.596149e+06,0.083292
47,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,60-69,297542.638838,5.614459e+06,0.052996
48,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Male,70+,522145.675064,1.343636e+07,0.038861
49,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Female,0-28 days,50469.243081,1.130016e+06,0.044662
50,Global 2019,Global,2019,20,letter,Infectious and parasitic diseases,Female,1-59 months,691051.821884,1.332074e+06,0.518779


In [ ]:
# Keep only the needed fields
project_df = df_clean[[
    "region",
    "sex",
    "age",
    "cause",
    "probability"
]].copy()

project_df = project_df.sort_values(
    by=["region", "sex", "age", "cause"]
).reset_index(drop=True)

project_df.head(30)

,region,sex,age,cause,probability
0,African Region,Female,0-28 days,Cardiovascular diseases,5.542940e-04
1,African Region,Female,0-28 days,Congenital anomalies,6.385851e-02
2,African Region,Female,0-28 days,Diabetes mellitus,0.000000e+00
3,African Region,Female,0-28 days,Digestive diseases,1.595196e-04
4,African Region,Female,0-28 days,"Endocrine, blood, immune disorders",4.681901e-04
5,African Region,Female,0-28 days,Genitourinary diseases,5.951322e-05
6,African Region,Female,0-28 days,Infectious and parasitic diseases,6.258531e-02
7,African Region,Female,0-28 days,Intentional injuries,7.604744e-06
8,African Region,Female,0-28 days,Malignant neoplasms,8.067134e-05
9,African Region,Female,0-28 days,Maternal conditions,0.000000e+00


In [ ]:
# check what categories are kept
sorted(project_df["cause"].unique())

['Cardiovascular diseases',
 'Congenital anomalies',
 'Diabetes mellitus',
 'Digestive diseases',
 'Endocrine, blood, immune disorders',
 'Genitourinary diseases',
 'Infectious and parasitic diseases',
 'Intentional injuries',
 'Malignant neoplasms',
 'Maternal conditions',
 'Mental and substance use disorders',
 'Musculoskeletal diseases',
 'Neonatal conditions',
 'Neurological conditions',
 'Nutritional deficiencies',
 'Oral conditions',
 'Other COVID-19 pandemic-related outcomes',
 'Other neoplasms',
 'Respiratory Infectious',
 'Respiratory diseases',
 'Sense organ diseases',
 'Skin diseases',
 'Sudden infant death syndrome',
 'Unintentional injuries']

In [ ]:
#  Shorten WHO region names
region_map = {
    "African Region": "Africa",
    "Region of the Americas": "Americas",
    "European Region": "Europe",
    "South-East Asia Region": "South-East Asia",
    "Western Pacific Region": "Western Pacific",
    "Eastern Mediterranean Region": "Eastern Mediterranean",
}

project_df = project_df[project_df["region"] != "Global"].copy()

project_df["region"] = project_df["region"].replace(region_map)

project_df = project_df.sort_values(
    by=["region", "sex", "age", "cause"]
).reset_index(drop=True)

project_df.head()

,region,sex,age,cause,probability
0,Africa,Female,0-28 days,Cardiovascular diseases,0.000554
1,Africa,Female,0-28 days,Congenital anomalies,0.063859
2,Africa,Female,0-28 days,Diabetes mellitus,0.000000
3,Africa,Female,0-28 days,Digestive diseases,0.000160
4,Africa,Female,0-28 days,"Endocrine, blood, immune disorders",0.000468


In [ ]:
project_df.to_csv("death_probabilities_2019_big_causes_no_global.csv", index=False)

In [ ]:
sorted(project_df["region"].unique())

['Africa',
 'Americas',
 'Eastern Mediterranean',
 'Europe',
 'South-East Asia',
 'Western Pacific']